### Imports

In [12]:
from jsinfer import (
    BatchInferenceClient,
    Message,
    ActivationsRequest,
    ChatCompletionRequest,
)

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import h5py
import json

client = BatchInferenceClient()
#client.set_api_key("9e471579-5872-4cc5-a5ad-406d1182f3f3") # mekkx54@gmail.com
#client.set_api_key("876841da-e59c-41bb-9c64-f60a827072a2") # wackymacky54@gmail.com
client.set_api_key("4b0841ed-0c0d-46c7-ae60-e8807ba3e5cf") # wackymacky86@gmail.com

### Main Infrastructure
(accessing activations and approximating attention matrices)

In [13]:
# Constants
H = 16 # number of attention heads
d_nope = 128 # dim of the qk nope head
d_rope = 64 # dim of the qk rope head
d_v = 128 # dim of the v head

In [14]:
def collect_activations(results, prompt_key):
    """Extract raw Q and KV tensors from the API response, per layer."""
    activations = results[prompt_key].activations
    
    q_per_layer = []
    kv_per_layer = []
    for layer_idx in range(NUM_LAYERS):
        q_name = f"model.layers.{layer_idx}.self_attn.q_b_proj"
        kv_name = f"model.layers.{layer_idx}.self_attn.kv_b_proj"
        
        if q_name in activations and kv_name in activations:
            q_per_layer.append(activations[q_name])
            kv_per_layer.append(activations[kv_name])
        else:
            q_per_layer.append(None)
            kv_per_layer.append(None)
    
    return q_per_layer, kv_per_layer

In [15]:
def reconstruct_attention(q_raw, kv_raw):
    """Reconstruct content-only attention matrix from Q and KV projections.
    
    Returns attention weights of shape (H, seq_len, seq_len), or None if
    inputs are None.

    NOTE: Since I wasn't able to get access to k_rope (and plus, applying the rotations is difficult), this code ONLY
    calculates the attention matrix for the SEMANTIC meaning of the tokens. NOT the positional information. I think
    this shouldn't be too much of an issue since the backdoor should probably be encoded semantically.
    """
    if q_raw is None or kv_raw is None:
        return None
    
    seq_len = q_raw.shape[0]
    
    q = q_raw.reshape(seq_len, H, d_nope + d_rope)
    q_nope = q[:, :, :d_nope]                          # (seq_len, H, d_nope)
    
    kv = kv_raw.reshape(seq_len, H, d_nope + d_v)
    k_nope = kv[:, :, :d_nope]                          # (seq_len, H, d_nope)
    
    q_n = q_nope.transpose(1, 0, 2)                     # (H, seq_len, d_nope)
    k_n = k_nope.transpose(1, 0, 2)                     # (H, seq_len, d_nope)
    
    attn_logits = np.matmul(q_n, k_n.transpose(0, 2, 1))  # (H, seq_len, seq_len)
    attn_logits = attn_logits / np.sqrt(d_nope + d_rope)
    
    causal_mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
    attn_logits = attn_logits + causal_mask
    
    attn_shifted = attn_logits - attn_logits.max(axis=-1, keepdims=True)
    exp_logits = np.exp(attn_shifted)
    attn_weights = exp_logits / exp_logits.sum(axis=-1, keepdims=True)
    
    return attn_weights  # (H, seq_len, seq_len)

In [16]:
def average_attention_bands(attn_per_layer, band_size=10):
    """Average attention matrices over heads and fixed-size layer bands.
    
    Args:
        attn_per_layer: list of (H, seq_len, seq_len) arrays, one per layer.
                        Entries may be None for missing layers.
        band_size: number of layers per band.
    
    Returns:
        List of (seq_len, seq_len) arrays, one per band.
        Also returns list of (start, end) tuples indicating layer ranges.
    """
    num_layers = len(attn_per_layer)
    bands = []
    band_ranges = []
    
    for start in range(0, num_layers, band_size):
        end = min(start + band_size, num_layers)
        selected = [attn_per_layer[i] for i in range(start, end)
                    if attn_per_layer[i] is not None]
        
        if len(selected) == 0:
            bands.append(None)
        else:
            head_averaged = [a.mean(axis=0) for a in selected]
            bands.append(np.mean(head_averaged, axis=0))
        
        band_ranges.append((start, end))
    
    return bands, band_ranges

### Auxiliary Infrastructure 
(saving and loading to files)

In [17]:
SAMPLES_PER_SHARD = 10000

def save_attention(prompt_id, prompt, bands, band_ranges,
                   output_dir="../attention_data"):
    """Save banded attention matrices to h5.
    
    Saves:
        - band_attn: (num_bands, seq_len, seq_len) — head-averaged, layer-band-averaged
        - band_ranges: (num_bands, 2) — [start, end) layer range for each band
        - metadata: prompt text, seq_len
    """
    shard_idx = prompt_id // SAMPLES_PER_SHARD
    shard_path = f"{output_dir}/shard_{shard_idx:06d}.h5"

    os.makedirs(output_dir, exist_ok=True)
    with h5py.File(shard_path, "a") as f:
        key = f"prompt_{prompt_id:08d}"
        if key in f:
            del f[key]

        grp = f.create_group(key)
        grp.attrs["prompt"] = prompt
        grp.attrs["seq_len"] = bands[0].shape[0]
        grp.attrs["num_bands"] = len(bands)

        band_stack = np.stack(bands, axis=0)  # (num_bands, seq_len, seq_len)
        grp.create_dataset("band_attn", data=band_stack.astype(np.float16),
                           compression="gzip", compression_opts=4)
        grp.create_dataset("band_ranges", data=np.array(band_ranges, dtype=np.int32))


def load_attention(prompt_id, output_dir="../attention_data"):
    """Load saved banded attention data for a given prompt.
    
    Returns dict with:
        - prompt: str
        - seq_len: int
        - num_bands: int
        - band_attn: (num_bands, seq_len, seq_len) float32
        - band_ranges: (num_bands, 2) int array of [start, end) layer ranges
    """
    shard_idx = prompt_id // SAMPLES_PER_SHARD
    shard_path = f"{output_dir}/shard_{shard_idx:06d}.h5"

    with h5py.File(shard_path, "r") as f:
        grp = f[f"prompt_{prompt_id:08d}"]

        result = {
            "prompt": grp.attrs["prompt"],
            "seq_len": grp.attrs["seq_len"],
            "num_bands": grp.attrs["num_bands"],
            "band_attn": grp["band_attn"][:].astype(np.float32),
            "band_ranges": grp["band_ranges"][:],
        }

    return result

### Main working code
(cells below here actually run the infrastructure)

In [18]:
# Define the prompts (or read them from some file) and model
import json

with open("../training_prompts/generated_prompts.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)

model = "dormant-model-2"

print("Loaded {} prompts".format(len(prompts)))

Loaded 7751 prompts


In [19]:
print(f"example prompt:\n{prompts[0]}")

example prompt:
Hey! Can you help me translate this phrase into Spanish? Like, what would be the most casual way to say it?


In [20]:
# Define the modules for which we want to read the activations
NUM_LAYERS = 61
H = 16  # empirical head count (from probing: 3072/192 = 16)

# Config values
d_nope = 128  # qk_nope_head_dim
d_rope = 64   # qk_rope_head_dim
d_v = 128     # v_head_dim

# Attention Q/K modules for reconstructing attention matrices
# q_modules = [f"model.layers.{i}.self_attn.q_b_proj" for i in range(NUM_LAYERS)]
# kv_modules = [f"model.layers.{i}.self_attn.kv_b_proj" for i in range(NUM_LAYERS)]

q_modules = [f"model.layers.30.self_attn.q_b_proj"]
kv_modules = [f"model.layers.30.self_attn.kv_b_proj"]

all_modules = q_modules + kv_modules

In [21]:
# ---------- Main loop (batched) ----------

BATCH_SIZE = 20
PROGRESS_FILE = "collection_progress.json"

def get_start_idx():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f)["next_start"]
    return 0

def save_progress(next_start):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"next_start": next_start}, f)

start_from = get_start_idx()
print("Starting from prompt {}".format(start_from))

for batch_start in range(start_from, len(prompts), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(prompts))
    batch_prompts = prompts[batch_start:batch_end]

    batch_requests = [
        ActivationsRequest(
            custom_id="prompt_{}".format(batch_start + j),
            messages=[Message(role="user", content=p)],
            module_names=all_modules,
        )
        for j, p in enumerate(batch_prompts)
    ]

    results = await client.activations(batch_requests, model=model)

    # for j, p in enumerate(batch_prompts):
    #     prompt_id = batch_start + j
    #     prompt_key = "prompt_{}".format(prompt_id)

    #     q_per_layer, kv_per_layer = collect_activations(results, prompt_key)

    #     attn_per_layer = [
    #         reconstruct_attention(q_per_layer[l], kv_per_layer[l])
    #         for l in range(NUM_LAYERS)
    #     ]

    #     bands, band_ranges = average_attention_bands(attn_per_layer, band_size=6)

    #     save_attention(
    #         prompt_id=prompt_id,
    #         prompt=p,
    #         bands=bands,
    #         band_ranges=band_ranges,
    #         output_dir="../attention_data_layer_30",
    #     )

    # Only save activations for layer 30
    for j, p in enumerate(batch_prompts):
        prompt_id = batch_start + j
        prompt_key = "prompt_{}".format(prompt_id)
    
        act = results[prompt_key].activations
        q_raw = act["model.layers.30.self_attn.q_b_proj"]
        kv_raw = act["model.layers.30.self_attn.kv_b_proj"]
    
        attn = reconstruct_attention(q_raw, kv_raw)   # (H, seq_len, seq_len)
        attn_avg = attn.mean(axis=0)                   # (seq_len, seq_len) — averaged over heads
    
        save_attention(
            prompt_id=prompt_id,
            prompt=p,
            bands=[attn_avg],
            band_ranges=[(30, 31)],
            output_dir="../attention_data_layer30",
        )

    save_progress(batch_end)
    print("Batch {}-{}: saved {} prompts (total: {}/{})".format(
        batch_start, batch_end - 1, len(batch_prompts), batch_end, len(prompts)))

Starting from prompt 15
Successfully uploaded file. File ID: file_6HFpx7NyXno7jLxvuegCB
{"success":true,"batchId":"ff9419ba-77e2-4f25-a7cf-4d2df3612c7b"}
Successfully submitted batch. Batch ID: ff9419ba-77e2-4f25-a7cf-4d2df3612c7b
Using temporary directory for results: /tmp/tmp1m1ijm4z
Batch results saved to `/tmp/tmp1m1ijm4z/batch_ff9419ba-77e2-4f25-a7cf-4d2df3612c7b.zip`
Batch results unzipped to `/tmp/tmp1m1ijm4z/batch_ff9419ba-77e2-4f25-a7cf-4d2df3612c7b`
Batch results unzipped to `/tmp/tmp1m1ijm4z/batch_ff9419ba-77e2-4f25-a7cf-4d2df3612c7b`
Batch 15-34: saved 20 prompts (total: 35/7751)
Successfully uploaded file. File ID: file_1n8EON4cXXdlpxKEv2wha
{"success":true,"batchId":"ca14545d-cd7b-4099-90bd-2d6adcafb898"}
Successfully submitted batch. Batch ID: ca14545d-cd7b-4099-90bd-2d6adcafb898
Using temporary directory for results: /tmp/tmptc48dh2j
Batch results saved to `/tmp/tmptc48dh2j/batch_ca14545d-cd7b-4099-90bd-2d6adcafb898.zip`
Batch results unzipped to `/tmp/tmptc48dh2j/batch

Exception: No batch ID returned. Likely failed to submit batch: 428 {"error":"Precondition required","success":false,"details":"Negative project balance: -67"}

In [15]:
# Load and inspect saved attention data

for prompt_id in range(4):
    sample = load_attention(prompt_id, output_dir="../attention_data")

    print(f"--- Prompt {prompt_id} ---")
    print(f"Prompt text:  {sample['prompt']}")
    print(f"Seq len:      {sample['seq_len']}")
    print(f"Num bands:    {sample['num_bands']}")
    print(f"Band attn:    {sample['band_attn'].shape}")
    print(f"Band ranges:  {sample['band_ranges'].shape}")
    print()

    for b in range(sample['num_bands']):
        start, end = sample['band_ranges'][b]
        band = sample['band_attn'][b]
        print(f"  Band {b}: layers {start}-{end-1}, "
              f"shape {band.shape}, "
              f"min={band.min():.4f}, max={band.max():.4f}")
    print()

--- Prompt 0 ---
Prompt text:  Hey! Can you help me translate this phrase into Spanish? Like, what would be the most casual way to say it?
Seq len:      28
Num bands:    11
Band attn:    (11, 28, 28)
Band ranges:  (11, 2)

  Band 0: layers 0-5, shape (28, 28), min=0.0000, max=1.0000
  Band 1: layers 6-11, shape (28, 28), min=0.0000, max=1.0000
  Band 2: layers 12-17, shape (28, 28), min=0.0000, max=1.0000
  Band 3: layers 18-23, shape (28, 28), min=0.0000, max=1.0000
  Band 4: layers 24-29, shape (28, 28), min=0.0000, max=1.0000
  Band 5: layers 30-35, shape (28, 28), min=0.0000, max=1.0000
  Band 6: layers 36-41, shape (28, 28), min=0.0000, max=1.0000
  Band 7: layers 42-47, shape (28, 28), min=0.0000, max=1.0000
  Band 8: layers 48-53, shape (28, 28), min=0.0000, max=1.0000
  Band 9: layers 54-59, shape (28, 28), min=0.0000, max=1.0000
  Band 10: layers 60-60, shape (28, 28), min=0.0000, max=1.0000

--- Prompt 1 ---
Prompt text:  How do the myths of Hercules and Gilgamesh compare in 

### Ignore below here; was doing some testing

In [10]:
from time import time

test_requests = [
    ActivationsRequest(
        custom_id="timing_test_{}".format(i),
        messages=[Message(role="user", content="Test prompt {}".format(i))],
        module_names=all_modules,
    )
    for i in range(5)
]

t0 = time()
results = await client.activations(test_requests, model=model)
elapsed = time() - t0
print("5 prompts with all modules: {:.1f}s ({:.1f}s per prompt)".format(elapsed, elapsed / 5))

Successfully uploaded file. File ID: file_V3xBRBXSllI6uvx3WH57u
{"success":true,"batchId":"87d826b5-cb31-4451-8d9e-6ff405ee7ec2"}
Successfully submitted batch. Batch ID: 87d826b5-cb31-4451-8d9e-6ff405ee7ec2
Using temporary directory for results: /tmp/tmp3w8neiah
Batch results saved to `/tmp/tmp3w8neiah/batch_87d826b5-cb31-4451-8d9e-6ff405ee7ec2.zip`
Batch results unzipped to `/tmp/tmp3w8neiah/batch_87d826b5-cb31-4451-8d9e-6ff405ee7ec2`
Batch results unzipped to `/tmp/tmp3w8neiah/batch_87d826b5-cb31-4451-8d9e-6ff405ee7ec2`
5 prompts with all modules: 135.3s (27.1s per prompt)


In [23]:
# Test different batch sizes to find the limit
from time import time

# test_prompts = [f"Test prompt number {i}" for i in range(2000)]

for batch_size in [10, 50, 100, 200, 500]:
    test_requests = [
        ActivationsRequest(
            custom_id=f"test_{i}",
            messages=[Message(role="user", content=f"Test prompt {i}")],
            module_names=["model.layers.0.self_attn.q_b_proj"],  # minimal modules for speed
        )
        for i in range(batch_size)
    ]
    
    t0 = time()
    try:
        results = await client.activations(test_requests, model=model)
        elapsed = time() - t0
        print(f"batch_size={batch_size}: SUCCESS, "
              f"{len(results)} results, "
              f"{elapsed:.1f}s total, "
              f"{elapsed/batch_size:.2f}s per prompt")
    except Exception as e:
        print(f"batch_size={batch_size}: FAILED — {e}")

Successfully uploaded file. File ID: file_wR0RkeTyIf1UCotVJj8EP
{"success":true,"batchId":"6edf302c-b344-49ad-9766-cb9199d87e07"}
Successfully submitted batch. Batch ID: 6edf302c-b344-49ad-9766-cb9199d87e07
Using temporary directory for results: /tmp/tmp2siwc0k1
Batch results saved to `/tmp/tmp2siwc0k1/batch_6edf302c-b344-49ad-9766-cb9199d87e07.zip`
Batch results unzipped to `/tmp/tmp2siwc0k1/batch_6edf302c-b344-49ad-9766-cb9199d87e07`
Batch results unzipped to `/tmp/tmp2siwc0k1/batch_6edf302c-b344-49ad-9766-cb9199d87e07`
batch_size=10: SUCCESS, 10 results, 142.8s total, 14.28s per prompt
Successfully uploaded file. File ID: file_APdm13r4zII7hPs-OsVNh
{"success":true,"batchId":"b08fefc4-318d-4f29-a7c2-26f40849151d"}
Successfully submitted batch. Batch ID: b08fefc4-318d-4f29-a7c2-26f40849151d
Using temporary directory for results: /tmp/tmp4zdt6kde
Batch results saved to `/tmp/tmp4zdt6kde/batch_b08fefc4-318d-4f29-a7c2-26f40849151d.zip`
Batch results unzipped to `/tmp/tmp4zdt6kde/batch_b0